In [0]:
import datetime

import pyspark
import pandas as pd

from py4j.protocol import Py4JJavaError

from config.settings import *

In [0]:
# Pull GCP credentials securely from Databricks Secrets and set them as configuration
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
SOURCE_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"
RAW_OUTPUT_DIR = f"gs://{gcp_project_id}/raw/airports"

In [0]:
# TODO: Add Logic to handle the exception (delete the output directory if it exists)
def upload_raw() -> pyspark.RDD:
    raw = pd.read_csv(SOURCE_URL)
    rows = raw.to_csv(index=False).splitlines()
    rdd = sc.parallelize(rows).filter(lambda row: row.strip())

    try:
        rdd.saveAsTextFile(RAW_OUTPUT_DIR)
        status = "SUCCESS"
    except Py4JJavaError as e:
        status = "FAILED"
        print(f"Error saving raw data to GCS: {str(e.java_exception)=}")
        raise
    
    return {
        "date": datetime.date.today().strftime("%Y-%m-%d"),
        "rdd_stored": RAW_OUTPUT_DIR,
        "status": status
    }

In [0]:
upload_raw()